# Moodle API Probe — Diagnostic Notebook

Walks every Moodle endpoint the ETL uses, prints accessibility status, and dumps a sample of the response shape.

**When to use:**
- Before running the ETL, to verify the token works
- When the ETL produces unexpected fallback data, to figure out which endpoint changed
- For industry partner to confirm production environment readiness 

**No data is saved** by this notebook — it's read-only diagnostic.


## 1. Setup

In [7]:
# =========================
# UNIVERSAL ENVIRONMENT SETUP + ETL IMPORTS
# =========================
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

def load_env_safely():
    current_path = Path.cwd()
    for i in range(5):
        env_path = current_path / ".env"
        if env_path.exists():
            print(f"✅ .env found at: {env_path}")
            load_dotenv(env_path, override=True)
            return env_path
        current_path = current_path.parent
    print("❌ .env file NOT found!")
    return None

env_path = load_env_safely()

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Now safe to import etl modules — env vars are loaded
from etl import schemas as S
from etl import io as etl_io
from etl import moodle_client as mc
from etl import transforms as T
from etl import risk as R
# Note: probe_moodle does NOT need mocks; hybrid_etl DOES
try:
    from etl import mocks as M
except ImportError:
    M = None

print("\n===== ENV STATUS =====")
print("Working directory:", Path.cwd())
print("Base URL:", mc.MOODLE_BASE_URL)
print("Token exists:", bool(mc.MOODLE_TOKEN))
print("Token preview:", mc.MOODLE_TOKEN[:6] + "..." if mc.MOODLE_TOKEN else "EMPTY")
print("Endpoint:", mc.MOODLE_ENDPOINT)
print("======================\n")

✅ .env found at: /Users/Kay/MDS651 Industry Capstone/.env

===== ENV STATUS =====
Working directory: /Users/Kay/MDS651 Industry Capstone/Notebook
Base URL: https://spi.nsw.edu.au/learn
Token exists: True
Token preview: 4daf5d...
Endpoint: https://spi.nsw.edu.au/learn/webservice/rest/server.php



## 2. Pre-flight token check

In [8]:
preflight = mc.preflight_check()
if not preflight['ok']:
    print()
    print(f"Failure mode: {preflight['failure_mode']}")
    print(f"Error: {preflight['error']}")
    print()
    print("Probable fix:")
    fixes = {
        'token_invalid': 'Refresh the Moodle Web Services token. Update MOODLE_TOKEN in .env.',
        'endpoint_blocked': 'Contact Moodle admin: token needs Web Services access.',
        'network_error': 'Check MOODLE_BASE_URL is reachable. VPN/firewall/proxy issue?',
    }
    print(f"  {fixes.get(preflight['failure_mode'], 'See error message above for hints.')}")


[moodle] Pre-flight FAILED (token_invalid): invalidtoken: Invalid token - token not found

Failure mode: token_invalid
Error: invalidtoken: Invalid token - token not found

Probable fix:
  Refresh the Moodle Web Services token. Update MOODLE_TOKEN in .env.


## 3. Tesing Each Endpoint

In [9]:
def probe(name, fn, *args):
    print(f"\\n--- {name} ---")
    try:
        result = fn(*args)
    except Exception as e:
        print(f"  EXCEPTION: {e}")
        return None
    if result.get('success'):
        data = result['data']
        if isinstance(data, list):
            print(f"  OK -> list with {len(data)} items")
            if data:
                print(f"  first item keys: {list(data[0].keys()) if isinstance(data[0], dict) else type(data[0]).__name__}")
        elif isinstance(data, dict):
            print(f"  OK -> dict with keys: {list(data.keys())[:10]}")
        else:
            print(f"  OK -> {type(data).__name__}")
        return data
    else:
        print(f"  FAILED: {result.get('error')}")
        return None

site = probe('core_webservice_get_site_info', mc.extract_site_info)
courses = probe('core_course_get_courses', mc.extract_all_courses)


\n--- core_webservice_get_site_info ---
  FAILED: invalidtoken: Invalid token - token not found
\n--- core_course_get_courses ---
  FAILED: invalidtoken: Invalid token - token not found


In [10]:
# Per-course probes
if isinstance(courses, list) and courses:
    course_ids = [c.get('id') for c in courses[:2] if c.get('id')]
elif isinstance(site, dict):
    course_ids = [c.get('id') for c in site.get('usercourses', [])[:2] if c.get('id')]
else:
    course_ids = []

print(f"\\nProbing courses: {course_ids}")

for cid in course_ids:
    participants = probe(f'core_enrol_get_enrolled_users (course {cid})',
                          mc.extract_course_participants, cid)
    assignments = probe(f'mod_assign_get_assignments (course {cid})',
                         mc.extract_assignments_for_course, cid)
    if isinstance(participants, list) and participants:
        first_uid = participants[0].get('id')
        if first_uid:
            grades = probe(f'gradereport_user_get_grade_items (user {first_uid}, course {cid})',
                           mc.extract_user_grade_items, first_uid, cid)


\nProbing courses: []


## 4. Summary

If you see a green path through all probes, hybrid_etl will run in real-mode and production_etl will succeed. If any probe fails, the failure mode tells you what to fix.

**Common token-fix steps for the SPI sandbox:**

1. Go to `https://spi.nsw.edu.au/learn/admin/tool/webservice/userselect.php`
2. Find your test user and either reset the token or copy the existing one
3. Paste into `.env` as `MOODLE_TOKEN=...`
4. Re-run this notebook to verify
